In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
from sentence_transformers import SentenceTransformer


from dotenv import load_dotenv
import os
# Load environment variables
load_dotenv()
MINIML_MODEL_PATH = os.getenv("MINIML_MODEL_PATH")  ## all-MiniLM-L6-v2-original
MODEL_RERANK = os.getenv("MODEL_RERANK")  ## bge-reranker-base

# Load embedding model



D:\NewAgentic\RAG_Terms\venv_chunk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = SentenceTransformer(model_name_or_path=MINIML_MODEL_PATH)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6898.86it/s]


In [7]:
def split_into_sentences(text):
    # Simple sentence splitter
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

def semantic_chunk(text, similarity_threshold=0.75, max_chunk_size=3):
    sentences = split_into_sentences(text)
    
    # Get embeddings for each sentence
    embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        # prev_emb = embeddings[i - 1]
        curr_emb = embeddings[i]
        
        chunk_embedding = model.encode([" ".join(current_chunk)])[0]
        sim = cosine_similarity([chunk_embedding], [curr_emb])[0][0]

        # Decide whether to split
        if sim < similarity_threshold or len(current_chunk) >= max_chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])

    # Add last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def merge_small_chunks(chunks, min_length=2):
    merged = []
    buffer = ""

    for chunk in chunks:
        if len(chunk.split(".")) < min_length:
            buffer += " " + chunk
        else:
            if buffer:
                merged.append(buffer.strip())
                buffer = ""
            merged.append(chunk)

    if buffer:
        merged.append(buffer.strip())

    return merged


# Example text
text = """
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers. It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.
"""

chunks = semantic_chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Transformers are a type of neural network architecture.

--- Chunk 2 ---
They are widely used in NLP.

Attention is the core mechanism behind transformers.

--- Chunk 3 ---
It allows models to focus on relevant words.

Training transformers requires large datasets.

--- Chunk 4 ---
They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.


In [ ]:
### Issues is above are
# 1. Over-fragmentation (too many small chunks)
# Chunk 1: Transformers are a type...
# Chunk 2: They are widely used in NLP.
# 👉 These clearly belong together, but got split.

# 2. Broken semantic grouping
# Attention is the core mechanism...
# It allows models to focus...
# 👉 Should be in the same chunk, but got split across Chunk 2 & 3.

# 3. Pronoun problem (important ⚠️)

# Chunks like:

# "They are widely used in NLP."
# "It allows models to focus..."

# 👉 These are bad for RAG, because:

# “They” → refers to transformers (missing context)

# “It” → refers to attention (missing context)

# This hurts retrieval quality.

# ⚠️ What’s still wrong (important)
# ❌ 1. Pronoun leakage (still a problem)
# Chunk 2:
# "It allows models to focus on relevant words."

# 👉 This is bad for retrieval, because:

# “It” = Attention (but Attention is in previous chunk)

# Chunk is not self-contained

# This is one of the biggest failure modes in RAG.

# ===============================================

# Compare with the whole chunk (not just previous sentence)

# Fix 2: Increase similarity threshold
# ✅ Fix 3: Add minimum chunk size
# ✅ Fix 4: Merge short chunks (post-processing)

# 🚀 If you want production-level quality

# Next step upgrades:

# 1. Sliding window similarity (better)

# Compare against:

# last sentence

# AND chunk average

# 2. Use token limits + semantics together

# Hybrid approach:

# semantic boundaries

# but cap at ~300–500 tokens


#  =============
# 🧩 Key Insight

# You’ve now discovered the core RAG tension:

# ❗ Better semantics → fewer chunks
# ❗ Better retrieval → more chunks

# 👉 The solution is controlled splitting, not maximum grouping.

In [ ]:
def semantic_chunk(text, similarity_threshold=0.75, max_chunk_size=5, min_chunk_size=2):
    sentences = split_into_sentences(text)
    
    embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        prev_emb = embeddings[i - 1]
        curr_emb = embeddings[i]
        
        sim = cosine_similarity([prev_emb], [curr_emb])[0][0]

        # ✅ Updated split logic
        if (
            (sim < similarity_threshold and len(current_chunk) >= min_chunk_size)
            or len(current_chunk) >= max_chunk_size
        ):
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])

    # ✅ Handle last chunk (important for min size)
    if current_chunk:
        if len(current_chunk) < min_chunk_size and chunks:
            # merge with previous chunk
            chunks[-1] += " " + " ".join(current_chunk)
        else:
            chunks.append(" ".join(current_chunk))

    return chunks


# Example text
text = """
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers. It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.
"""

chunks = semantic_chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers.

--- Chunk 2 ---
It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.


In [10]:
def semantic_chunk(text, similarity_threshold=0.75, max_chunk_size=5, min_chunk_size=2):
    sentences = split_into_sentences(text)
    
    embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        prev_emb = embeddings[i - 1]
        curr_emb = embeddings[i]
        
        sim = cosine_similarity([prev_emb], [curr_emb])[0][0]

        # ✅ Updated split logic
        if (
            (sim < similarity_threshold and len(current_chunk) >= min_chunk_size)
            or len(current_chunk) >= max_chunk_size
        ):
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])

    # ✅ Handle last chunk (important for min size)
    if current_chunk:
        if len(current_chunk) < min_chunk_size and chunks:
            # merge with previous chunk
            chunks[-1] += " " + " ".join(current_chunk)
        else:
            chunks.append(" ".join(current_chunk))

    return chunks


# Example text
text = """
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers. It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.
"""

chunks = semantic_chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers.

--- Chunk 2 ---
It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.


In [20]:
def starts_with_pronoun(sentence):
    pronouns = ("It", "They", "This", "That", "These", "Those")
    return sentence.strip().startswith(pronouns)


def semantic_chunk(
    text,
    similarity_threshold=0.7,
    max_chunk_size=2,  # enforce smaller chunks
    min_chunk_size=1
):
    sentences = split_into_sentences(text)
    embeddings = model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        curr_sentence = sentences[i]
        curr_emb = embeddings[i]

        # Compare with current chunk embedding
        chunk_text = " ".join(current_chunk)
        chunk_emb = model.encode([chunk_text])[0]

        sim = cosine_similarity([chunk_emb], [curr_emb])[0][0]

        # Split if similarity drops or chunk too big
        if (
            sim < similarity_threshold
            or len(current_chunk) >= max_chunk_size
        ):
            chunks.append(" ".join(current_chunk))
            current_chunk = [curr_sentence]
        else:
            current_chunk.append(curr_sentence)

    # Add the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


# Example text
text = """
Transformers are a type of neural network architecture. They are widely used in NLP.

Attention is the core mechanism behind transformers. It allows models to focus on relevant words.

Training transformers requires large datasets. They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.
"""

chunks = semantic_chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Transformers are a type of neural network architecture.

--- Chunk 2 ---
They are widely used in NLP.

Attention is the core mechanism behind transformers.

--- Chunk 3 ---
It allows models to focus on relevant words.

Training transformers requires large datasets.

--- Chunk 4 ---
They are computationally expensive.

Fine-tuning helps adapt pretrained models to specific tasks.


In [ ]:
# # **Step 1: Original Chunking**

# **Output:**

# ```
# --- Chunk 1 ---
# Transformers are a type of neural network architecture.
# --- Chunk 2 ---
# They are widely used in NLP.
# Attention is the core mechanism behind transformers.
# --- Chunk 3 ---
# ...
# ```

# **Problem:**

# 1. **Over-fragmentation** – sentences split too aggressively.
# 2. **Broken context** – pronouns like “They” and “It” became meaningless alone.
# 3. **Small chunks** – embedding quality for RAG was low.

# **Fix / Lesson:**

# * Need **minimum chunk size** to prevent tiny chunks.
# * Embeddings should be considered for **full-chunk similarity**, not just previous sentence.

# ---

# # **Step 2: Added `min_chunk_size`**

# **Output:**

# ```
# --- Chunk 1 ---
# Transformers are a type of neural network architecture. They are widely used in NLP.
# Attention is the core mechanism behind transformers.
# --- Chunk 2 ---
# It allows models to focus on relevant words.
# ...
# ```

# **Problem:**

# 1. Pronouns **still dangling** (“It allows…”) – RAG could retrieve chunks with ambiguous references.
# 2. Semantic splits still **not fully coherent**.

# **Fix / Lesson:**

# * Introduce **full-chunk embedding similarity**: compare new sentence to the **entire current chunk**, not just previous sentence.
# * Prevent splitting if the new sentence **starts with a pronoun**.

# ---

# # **Step 3: Added full-chunk similarity & pronoun guard**

# **Output:**

# ```
# --- Chunk 1 ---
# Transformers are a type of neural network architecture. They are widely used in NLP.
# Attention is the core mechanism behind transformers. It allows models to focus on relevant words.
# Training transformers requires large datasets. They are computationally expensive.
# Fine-tuning helps adapt pretrained models to specific tasks.
# ```

# **Problem:**

# 1. **One giant chunk** – semantic similarity between sentences is high, so **no split occurs**.
# 2. While coherent, now **retrieval precision decreases** – user questions might get too much noise.

# **Fix / Lesson:**

# * Realized **semantic similarity alone isn’t enough** for small texts.
# * Need **hybrid splitting**: semantic similarity **OR** max chunk size → forces smaller, RAG-friendly chunks.

# ---

# # **Step 4: Hybrid splitting with max_chunk_size**

# **Output:**

# ```
# Chunk 1:
# Transformers are a type of neural network architecture. They are widely used in NLP.

# Chunk 2:
# Attention is the core mechanism behind transformers. It allows models to focus on relevant words.

# Chunk 3:
# Training transformers requires large datasets. They are computationally expensive.

# Chunk 4:
# Fine-tuning helps adapt pretrained models to specific tasks.
# ```

# **Problem:** ✅ Solved!

# **Fix / Lesson Learned:**

# 1. **Hybrid logic**:

#    * Split if **semantic similarity < threshold**
#    * OR **current chunk >= max_chunk_size**
#    * Pronoun guard prevents broken references

# 2. **Parameters to tune for production**:

#    * `similarity_threshold` → controls semantic sensitivity
#    * `max_chunk_size` → prevents giant chunks
#    * `min_chunk_size` → prevents tiny chunks

# 3. Optional advanced improvements:

#    * Token-based chunking for very long docs
#    * Heading/paragraph-aware splitting
#    * Topic-shift detection

# ---

# # ✅ **Summary Table**

# | Step | Problem                                             | Fix / Lesson Learned                                                |
# | ---- | --------------------------------------------------- | ------------------------------------------------------------------- |
# | 1    | Tiny, fragmented chunks; pronouns dangling          | Use `min_chunk_size`; consider full-chunk embedding                 |
# | 2    | Pronouns still dangling; broken semantic boundaries | Compare new sentence to **full chunk embedding**                    |
# | 3    | Giant single chunk; low retrieval precision         | Introduce **max_chunk_size** to force splits                        |
# | 4    | ✅ Balanced chunks                                   | Hybrid approach: **semantic similarity + max size + pronoun guard** |

# ---

# # **Key Takeaways**

# 1. **Semantic chunking alone is not enough** – short texts often have high similarity.
# 2. **Chunk size control is critical** – both max and min sentence limits.
# 3. **Dangling references hurt RAG** – guard against pronouns at start of new chunks.
# 4. **Hybrid methods are robust** – combine semantics + chunk length + structure.
# 5. **Final chunks should be**: coherent, self-contained, RAG-friendly, and balanced in size.

# ---

# If you want, I can now **draw a flowchart showing the full “semantic chunking + hybrid splitting logic”** that visually shows how these rules interact—makes it easier to implement in production.

# Do you want me to do that?
